# Đề án 05 – Giọng điệu văn bản báo cáo và phản ứng thị trường (Mỹ + Việt Nam)
Notebook chạy **từ dữ liệu đã xử lý** (`data/<thị trường>/processed/`). Muốn tải lại từ đầu: `python run_all.py --market both`.

**Câu hỏi:** giọng điệu (tone) của 10-K / Thông điệp HĐQT có tác động tích cực hay tiêu cực lên CAR[T, T+3] không – văn bản có mang thông tin cho thị trường không?

In [ ]:
import sys, os, pathlib, pandas as pd
from IPython.display import display, Image, Markdown
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src')); os.chdir(ROOT)
from analysis import a01_tone, a02_event, a03_regress, a04_figures
MARKETS = [m for m in ['us', 'vn'] if (ROOT / 'data' / m / 'processed' / 'docs.csv').exists()]
MARKETS

## 1. Chạy lại phần phân tích (tone → sự kiện → hồi quy → hình)

In [ ]:
for m in MARKETS:
    for step in (a01_tone, a02_event, a03_regress, a04_figures):
        step.main(m)

## Kết quả: Mỹ – 10-K

In [ ]:
m = 'us'
if m in MARKETS:
    out = ROOT / 'outputs' / m
    display(Markdown('**Mục tiêu 1 – thống kê tone**'), pd.read_csv(ROOT/'data'/m/'processed'/'tone_panel.csv').filter(regex='^(fin|gen)_').describe().T.round(4))
    display(Markdown('**Mục tiêu 2 – từ điển tổng quát gán sai**'), Markdown('```\n'+(out/'dictionary_comparison.txt').read_text(encoding='utf-8')+'\n```'))
    display(pd.read_csv(out/'misclassified_general_neg.csv').head(15), Image(str(out/'fig2_misclassified.png'), width=560))
    display(Markdown('**Mục tiêu 3 – CAR có khác 0 không?**'), pd.read_csv(out/'car_tests.csv'))
    display(Markdown('**CAR[0,3] theo nhóm giọng điệu**'), pd.read_csv(out/'car_by_tone.csv'), Image(str(out/'fig3_caar_by_tone.png'), width=560))
    for f in ['regression_main', 'regression_fama_macbeth', 'regression_robustness', 'assumption_tests']:
        display(Markdown(f'**{f}**'), pd.read_csv(out/f'{f}.csv'))


## Kết quả: Việt Nam – Thông điệp HĐQT

In [ ]:
m = 'vn'
if m in MARKETS:
    out = ROOT / 'outputs' / m
    display(Markdown('**Mục tiêu 1 – thống kê tone**'), pd.read_csv(ROOT/'data'/m/'processed'/'tone_panel.csv').filter(regex='^(fin|gen)_').describe().T.round(4))
    display(Markdown('**Mục tiêu 2 – từ điển tổng quát gán sai**'), Markdown('```\n'+(out/'dictionary_comparison.txt').read_text(encoding='utf-8')+'\n```'))
    display(pd.read_csv(out/'misclassified_general_neg.csv').head(15), Image(str(out/'fig2_misclassified.png'), width=560))
    display(Markdown('**Mục tiêu 3 – CAR có khác 0 không?**'), pd.read_csv(out/'car_tests.csv'))
    display(Markdown('**CAR[0,3] theo nhóm giọng điệu**'), pd.read_csv(out/'car_by_tone.csv'), Image(str(out/'fig3_caar_by_tone.png'), width=560))
    for f in ['regression_main', 'regression_fama_macbeth', 'regression_robustness', 'assumption_tests']:
        display(Markdown(f'**{f}**'), pd.read_csv(out/f'{f}.csv'))


## Diễn giải nhanh
- **T3 − T1 dương, có ý nghĩa** và hệ số `fin_neg_z` âm → giọng điệu mang thông tin.
- **M4 đối đầu**: nếu `fin_neg_z` giữ ý nghĩa còn `gen_neg_z` mất → từ điển tài chính vượt trội (Mục tiêu 2).
- **Placebo** không có ý nghĩa → kết quả không phải do ngẫu nhiên/xu hướng chung.